In [ ]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [11]:
# Initialize sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

In [ ]:
# For Melbourne, use: CBD, beaches (St Kilda), airports, etc.
MELBOURNE_CBD = (-37.8136, 144.9631)
ST_KILDA_BEACH = (-37.8678, 144.9816)
MELBOURNE_AIRPORT = (-37.6690, 144.8410)
# Add highway entrances if known, e.g., CityLink entry
HIGHWAY_ENTRANCES = [
    (-37.80, 144.95), 
    (-37.82, 144.97),
]

In [15]:
# GEOGRAPHIC ENGINEERING
def compute_distances(df):
    df = df.copy()
    df['dist_to_cbd'] = df.apply(
        lambda row: geodesic((row['latitude'], row['longitude']), MELBOURNE_CBD).km, axis=1
    )
    df['dist_to_beach'] = df.apply(
        lambda row: geodesic((row['latitude'], row['longitude']), ST_KILDA_BEACH).km, axis=1
    )
    df['dist_to_airport'] = df.apply(
        lambda row: geodesic((row['latitude'], row['longitude']), MELBOURNE_AIRPORT).km, axis=1
    )
    # Min distance to any highway entrance
    df['dist_to_highway'] = df.apply(
        lambda row: min(
            [geodesic((row['latitude'], row['longitude']), hw).km for hw in HIGHWAY_ENTRANCES]
        ), axis=1
    )
    return df

In [16]:
# AMENITY ENGINEERING
# Define luxury vs. basic amenities (customize for Melbourne/AU context)
LUXURY_AMENITIES = {
    'pool', 'hot tub', 'fireplace', 'bbq', 'gym', 'beachfront', 'waterfront',
    'patio', 'balcony', 'private entrance', 'elevator', 'doorman', 'washer', 'dryer'
}
BASIC_AMENITIES = {
    'wifi', 'kitchen', 'tv', 'free parking', 'heating', 'air conditioning',
    'essentials', 'luggage dropoff', 'lockbox'
}

def engineer_amenities(df):
    df = df.copy()
    
    def count_amenities(amenity_list, target_set):
        if not isinstance(amenity_list, (list, set)):
            return 0
        return len(set(amenity_list) & target_set)
    
    df['cnt_luxury_amen'] = df['amenities'].apply(lambda x: count_amenities(x, LUXURY_AMENITIES))
    df['cnt_basic_amen'] = df['amenities'].apply(lambda x: count_amenities(x, BASIC_AMENITIES))
    df['total_amenities'] = df['amenities'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    
    # Ratio features
    df['luxury_amen_ratio'] = df['cnt_luxury_amen'] / (df['total_amenities'] + 1)
    
    return df

In [17]:
# TEXT SENTIMENT FEATURES
def get_sentiment(text):
    if pd.isna(text) or text.strip() == "":
        return 0.0
    scores = analyzer.polarity_scores(text)
    return scores['compound']  # Range: [-1, 1]

def engineer_text_features(df):
    df = df.copy()
    for col in ['description', 'neighborhood_overview']:
        if col in df.columns:
            df[f'{col}_sentiment'] = df[col].apply(get_sentiment)
            df[f'{col}_length'] = df[col].fillna("").apply(len)
        else:
            df[f'{col}_sentiment'] = 0.0
            df[f'{col}_length'] = 0
    return df

In [18]:
# HOST & LISTING BEHAVIORAL FEATURES
def engineer_host_features(df):
    df = df.copy()
    
    # Host tenure (already partially done, but ensure it's in days)
    if 'host_since_days' not in df.columns and 'host_since' in df.columns:
        ref_date = pd.Timestamp('2025-06-01')  # Adjust to your data scrape date
        df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
        df['host_since_days'] = (ref_date - df['host_since']).dt.days.fillna(0)
    
    # Professional host proxy
    df['is_multi_listings'] = (df['host_total_listings_count'] > 1).astype(int)
    df['is_professional_host'] = (df['host_total_listings_count'] >= 3).astype(int)
    
    # Response quality composite
    df['host_response_rate'] = pd.to_numeric(df['host_response_rate'], errors='coerce').fillna(0)
    df['host_acceptance_rate'] = pd.to_numeric(df['host_acceptance_rate'], errors='coerce').fillna(0)
    df['host_responsiveness_score'] = (
        df['host_response_rate'] * 0.5 + 
        df['host_acceptance_rate'] * 0.3 + 
        df['host_is_superhost'] * 20  # boost superhosts
    )
    
    return df

In [19]:
# INTERACTION & CAPACITY FEATURES
def engineer_interactions(df):
    df = df.copy()
    
    # Capacity proxies
    df['guests_per_bedroom'] = df['accommodates'] / (df['bedrooms'] + 1)
    df['bathrooms_per_bedroom'] = df['bathrooms'] / (df['bedrooms'] + 1)
    
    # Review activity
    df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
    df['is_low_review_count'] = (df['number_of_reviews'] <= 5).astype(int)
    
    return df

In [ ]:
# MAIN FUNCTION
def feature_engineering(df):
    print("Starting feature engineering...")
    
    # Ensure amenities is list (if stored as string)
    if df['amenities'].dtype == 'object':
        df['amenities'] = df['amenities'].apply(
            lambda x: eval(x) if isinstance(x, str) else x if isinstance(x, list) else []
        )
    
    df = compute_distances(df)
    print("Geographic features added")
    
    df = engineer_amenities(df)
    print("Amenity features added")
    
    df = engineer_text_features(df)
    print("Text sentiment features added")
    
    df = engineer_host_features(df)
    print("Host behavior features added")
    
    df = engineer_interactions(df)
    print("Interaction features added")
    
    # Drop raw text & high-cardinality columns if still present
    cols_to_drop = [
        'description', 'neighborhood_overview', 'amenities',
        'host_since', 'room_type', 'property_type'
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
    
    print(f"Final feature count: {df.shape[1]}")
    return df